# Module 16: NumPy and Pandas for Bioinformatics

**Estimated time: 90-120 minutes**

---

## Learning Objectives

- Create and manipulate NumPy arrays for numerical biological data
- Perform vectorized operations instead of Python loops
- Use broadcasting, indexing, and statistical functions
- Build Pandas Series and DataFrames for tabular biological data
- Read and write CSV, TSV, and Excel files
- Filter, group, and merge biological datasets

### Why NumPy and Pandas?

Bioinformatics data is inherently numerical and tabular: gene expression matrices, variant call tables, sequence feature annotations, sample metadata sheets. NumPy gives you fast array math (10-100x faster than Python loops), and Pandas gives you labeled tables that behave like spreadsheets you can program.

---

---

[← Previous: Module 15: Error Handling and Debugging](../../Tier_1_Python_for_Bioinformatics/15_Error_Handling/01_error_handling.ipynb) | [Next: Module 17: Data Wrangling with Pandas →](../../Tier_1_Python_for_Bioinformatics/17_Data_Wrangling/01_data_wrangling.ipynb)

---

In [ ]:
import numpy as np
import pandas as pd

# Display settings
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)

---

## Part 1: NumPy Fundamentals

NumPy's core object is the **ndarray** -- an n-dimensional array of elements that all share the same type. Unlike Python lists, arrays support vectorized operations: you write the operation once and it applies to every element without an explicit loop.

### 1.1 Creating Arrays

In [ ]:
# From a Python list
gc_values = np.array([0.42, 0.51, 0.48, 0.55, 0.44])
print("GC content per gene:", gc_values)
print("Type:", type(gc_values))
print("Dtype:", gc_values.dtype)

In [ ]:
# Convenience constructors
zeros = np.zeros(10)                  # 10 zeros
ones = np.ones((3, 4))               # 3x4 matrix of ones
positions = np.arange(0, 1000, 100)  # like range(), but returns array
fractions = np.linspace(0, 1, 5)     # 5 evenly spaced values in [0, 1]
empty_matrix = np.full((3, 3), np.nan)  # 3x3 filled with NaN

print("zeros:", zeros)
print("ones shape:", ones.shape)
print("positions:", positions)
print("fractions:", fractions)
print("empty_matrix:\n", empty_matrix)

In [ ]:
# 2D array -- a gene expression matrix (rows=genes, columns=samples)
expression = np.array([
    [120, 135, 128],   # Gene A across 3 samples
    [ 45,  50,  48],   # Gene B
    [300, 280, 310],   # Gene C
])

print("Shape:", expression.shape)    # (rows, cols)
print("Dimensions:", expression.ndim)
print("Total elements:", expression.size)
print("Matrix:\n", expression)

### 1.2 Indexing and Slicing

NumPy arrays use comma-separated indices for multiple dimensions: `arr[row, col]`. Slices, integer arrays, and boolean arrays all work as indices.

In [ ]:
# 1D indexing
reads = np.array([63, 0, 17, 250, 89, 42, 130])

print("First element:", reads[0])
print("Last element:", reads[-1])
print("Slice [2:5]:", reads[2:5])
print("Every other:", reads[::2])

In [ ]:
# 2D indexing on the expression matrix
print("Row 0 (Gene A):", expression[0])        # entire row
print("Column 1 (Sample 2):", expression[:, 1])  # entire column
print("Gene C, Sample 3:", expression[2, 2])    # single element
print("Genes A-B, Samples 1-2:\n", expression[:2, :2])  # submatrix

In [ ]:
# Boolean indexing -- filter without loops
print("Reads > 50:", reads[reads > 50])
print("Non-zero reads:", reads[reads != 0])

# Boolean mask
high_expr_mask = reads > 100
print("Mask:", high_expr_mask)
print("High-count reads:", reads[high_expr_mask])

### 1.3 Vectorized Operations

Every arithmetic operation on NumPy arrays is **vectorized** -- it applies element-by-element with no Python loop. This is both faster and more readable.

In [ ]:
# Gene coordinates: start and stop positions
starts = np.array([11869, 14404, 17369, 29554, 30366, 34554, 52473])
stops  = np.array([14409, 29570, 17436, 31109, 30503, 36081, 53312])

# Calculate gene lengths -- vectorized, no loop needed
lengths = stops - starts + 1
print("Gene lengths:", lengths)

# Log-transform expression counts
counts = np.array([63, 119, 17, 250, 89, 42, 130], dtype=float)
log_counts = np.log2(counts + 1)  # +1 to avoid log(0)
print("Log2 counts:", np.round(log_counts, 2))

In [ ]:
# RPKM normalization (Reads Per Kilobase per Million mapped reads)
def compute_rpkm(counts, lengths):
    """Normalize raw read counts to RPKM."""
    rpm = counts * (1_000_000 / np.sum(counts))   # per million
    rpkm = rpm * (1_000 / lengths)                 # per kilobase
    return rpkm

rpkm = compute_rpkm(counts, lengths)
print("RPKM values:", np.round(rpkm, 2))

### 1.4 Broadcasting

When arrays have different but compatible shapes, NumPy **broadcasts** the smaller array to match. This lets you write concise code for operations across rows or columns.

In [ ]:
# Expression matrix: 4 genes x 3 samples
expr = np.array([
    [120, 135, 128],
    [ 45,  50,  48],
    [300, 280, 310],
    [ 10,  12,   9],
], dtype=float)

# Normalize each sample (column) to its total -- broadcasting divides
# a (4,3) matrix by a (1,3) row vector
sample_totals = expr.sum(axis=0, keepdims=True)  # shape (1, 3)
cpm = expr / sample_totals * 1_000_000

print("Sample totals:", sample_totals.flatten())
print("CPM (counts per million):\n", np.round(cpm, 1))

In [ ]:
# Z-score each gene (row) across samples
gene_means = expr.mean(axis=1, keepdims=True)  # shape (4, 1)
gene_stds  = expr.std(axis=1, keepdims=True)

z_scores = (expr - gene_means) / gene_stds
print("Z-scores:\n", np.round(z_scores, 2))

### 1.5 Statistical Functions

In [ ]:
# Simulated gene expression for 100 genes
np.random.seed(42)
data = np.random.lognormal(mean=5, sigma=1, size=100)

print(f"Mean:   {np.mean(data):.2f}")
print(f"Median: {np.median(data):.2f}")
print(f"Std:    {np.std(data):.2f}")
print(f"Min:    {np.min(data):.2f}")
print(f"Max:    {np.max(data):.2f}")
print(f"25th percentile: {np.percentile(data, 25):.2f}")
print(f"75th percentile: {np.percentile(data, 75):.2f}")

In [ ]:
# axis parameter: compute along rows or columns
print("Per-gene mean (across samples):", np.round(expr.mean(axis=1), 1))
print("Per-sample mean (across genes):", np.round(expr.mean(axis=0), 1))
print("Grand mean:", np.round(expr.mean(), 1))

### 1.6 Reshape and Flatten

In [ ]:
# A 2x6 matrix reshaped into 3x4, 6x2, or flattened
matrix = np.arange(12).reshape(2, 6)
print("Original (2x6):\n", matrix)
print("Reshaped (3x4):\n", matrix.reshape(3, 4))
print("Flattened:", matrix.flatten())

### 1.7 Linear Algebra Basics

In [ ]:
# Dot product: correlation-like score between two gene profiles
gene_a = np.array([1.2, 3.4, 2.1, 5.0])
gene_b = np.array([1.1, 3.5, 2.0, 4.8])

dot = np.dot(gene_a, gene_b)
print(f"Dot product: {dot:.2f}")

# Correlation coefficient
corr = np.corrcoef(gene_a, gene_b)
print(f"Pearson r: {corr[0, 1]:.4f}")

In [ ]:
# Matrix multiplication: transform a count matrix
# rows = genes, cols = samples
count_matrix = np.array([[10, 20], [30, 40], [50, 60]])

# A 2x2 normalization/rotation matrix
transform = np.array([[0.5, 0.5], [0.5, -0.5]])

result = count_matrix @ transform  # or np.matmul(count_matrix, transform)
print("Original (3x2):\n", count_matrix)
print("After transform (3x2):\n", result)

### 1.8 Biological Application: Position Weight Matrix

A PWM represents the frequency of each nucleotide at each position in a set of aligned binding-site sequences. It is stored as a 4 x L NumPy array (rows = A, T, G, C; columns = positions).

In [ ]:
def build_pwm(sequences):
    """Build a Position Weight Matrix from aligned sequences."""
    mapping = {'A': 0, 'T': 1, 'G': 2, 'C': 3}
    seq_len = len(sequences[0])
    counts = np.zeros((4, seq_len))
    for seq in sequences:
        for i, nuc in enumerate(seq):
            counts[mapping[nuc], i] += 1
    return counts / len(sequences)


def score_sequence(pwm, sequence):
    """Score a sequence against a PWM (sum of log-odds)."""
    mapping = {'A': 0, 'T': 1, 'G': 2, 'C': 3}
    score = 0.0
    for i, nuc in enumerate(sequence):
        freq = pwm[mapping[nuc], i]
        score += np.log2(freq / 0.25) if freq > 0 else -10
    return score


# TATA-box motif examples
tata_seqs = [
    "TATAAAG",
    "TATAAAT",
    "TATAAAA",
    "TATAAAT",
    "TATAAAG",
    "TATAAAC",
]

pwm = build_pwm(tata_seqs)
print("PWM (rows: A, T, G, C):")
for i, nuc in enumerate("ATGC"):
    print(f"  {nuc}: {np.round(pwm[i], 2)}")

# Score a candidate sequence
print(f"\nScore 'TATAAAA': {score_sequence(pwm, 'TATAAAA'):.2f}")
print(f"Score 'GCGCGCG': {score_sequence(pwm, 'GCGCGCG'):.2f}")

### 1.9 Biological Application: Sliding Window GC Content

In [ ]:
def sliding_gc(sequence, window=50):
    """Calculate GC content in a sliding window using NumPy cumsum."""
    gc_binary = np.array([1 if n in 'GC' else 0 for n in sequence])
    cumsum = np.insert(np.cumsum(gc_binary), 0, 0)
    window_sums = cumsum[window:] - cumsum[:-window]
    return window_sums / window * 100


# A synthetic sequence with variable GC content
np.random.seed(7)
low_gc = ''.join(np.random.choice(['A', 'T', 'G', 'C'], 100, p=[0.35, 0.35, 0.15, 0.15]))
high_gc = ''.join(np.random.choice(['A', 'T', 'G', 'C'], 100, p=[0.15, 0.15, 0.35, 0.35]))
seq = low_gc + high_gc

gc = sliding_gc(seq, window=20)
print(f"Sequence length: {len(seq)}")
print(f"GC values computed: {len(gc)}")
print(f"Mean GC in first half:  {gc[:90].mean():.1f}%")
print(f"Mean GC in second half: {gc[90:].mean():.1f}%")

---

## Part 2: Pandas Fundamentals

Pandas extends NumPy with **labeled axes** (row index and column names) and rich I/O. Its two main structures are:

| Structure | Description |
|-----------|-------------|
| **Series** | A 1D labeled array (like a column in a spreadsheet) |
| **DataFrame** | A 2D labeled table (rows and columns) |

### 2.1 Series

In [ ]:
# From a list (default integer index)
gc_series = pd.Series([0.42, 0.51, 0.48, 0.55, 0.44],
                      name='gc_content')
print(gc_series)
print("\nType:", type(gc_series))

In [ ]:
# From a dictionary -- keys become the index
complement = pd.Series({'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'})
print(complement)
print("\nComplement of G:", complement['G'])
print("Values:", complement.values)

In [ ]:
# Vectorized operations on Series
gene_lengths = pd.Series({'BRCA1': 7088, 'TP53': 2512, 'EGFR': 5616, 'MYC': 2357})
print("Lengths in kilobases:")
print(gene_lengths / 1000)

### 2.2 Creating DataFrames

In [ ]:
# From a dictionary -- each key becomes a column
genes_df = pd.DataFrame({
    'gene':       ['BRCA1', 'TP53', 'EGFR', 'MYC', 'KRAS'],
    'chromosome':  ['17',    '17',   '7',    '8',   '12'],
    'length_bp':  [7088,    2512,   5616,   2357,  5764],
    'gc_content': [0.423,   0.512,  0.487,  0.551, 0.448],
    'biotype':    ['protein_coding'] * 5,
})

print(genes_df)
print(f"\nShape: {genes_df.shape}")
print(f"Columns: {genes_df.columns.tolist()}")

In [ ]:
# Quick overview methods
print("--- dtypes ---")
print(genes_df.dtypes)
print("\n--- describe ---")
print(genes_df.describe())
print("\n--- info ---")
genes_df.info()

### 2.3 Indexing with loc and iloc

| Accessor | Lookup by | Example |
|----------|-----------|--------|
| `.loc[]` | **label** (index/column name) | `df.loc[0, 'gene']` |
| `.iloc[]` | **integer position** | `df.iloc[0, 0]` |

In [ ]:
# Column access
print("Single column (Series):")
print(genes_df['gene'])

print("\nMultiple columns (DataFrame):")
print(genes_df[['gene', 'gc_content']])

In [ ]:
# loc -- label-based
print("Row 0:")
print(genes_df.loc[0])

print(f"\ngenes_df.loc[0, 'gene'] = {genes_df.loc[0, 'gene']}")

print("\nRows 1-3, columns gene and length_bp:")
print(genes_df.loc[1:3, ['gene', 'length_bp']])

In [ ]:
# iloc -- integer position
print("First 2 rows, first 3 columns:")
print(genes_df.iloc[:2, :3])

print(f"\nLast row, last column: {genes_df.iloc[-1, -1]}")

### 2.4 Filtering Rows

In [ ]:
# Boolean condition
long_genes = genes_df[genes_df['length_bp'] > 5000]
print("Genes longer than 5000 bp:")
print(long_genes[['gene', 'length_bp']])

In [ ]:
# Multiple conditions -- use & (and), | (or), ~ (not) with parentheses
result = genes_df[(genes_df['chromosome'] == '17') & (genes_df['gc_content'] > 0.45)]
print("Chr 17 genes with GC > 45%:")
print(result[['gene', 'chromosome', 'gc_content']])

In [ ]:
# The .query() method -- same filter, more readable
result2 = genes_df.query('chromosome == "17" and gc_content > 0.45')
print(result2[['gene', 'chromosome', 'gc_content']])

### 2.5 Sorting

In [ ]:
print("Sorted by GC content (descending):")
print(genes_df.sort_values('gc_content', ascending=False)[['gene', 'gc_content']])

### 2.6 Adding and Transforming Columns

In [ ]:
# Vectorized column creation
genes_df['length_kb'] = genes_df['length_bp'] / 1000

# Categorize with apply
genes_df['gc_category'] = genes_df['gc_content'].apply(
    lambda x: 'high' if x > 0.50 else ('medium' if x > 0.45 else 'low')
)

print(genes_df[['gene', 'length_kb', 'gc_content', 'gc_category']])

### 2.7 Reading Files: CSV, TSV, Excel

Pandas can read many formats. The most common in bioinformatics:

```python
# CSV
df = pd.read_csv('expression.csv')

# TSV (tab-separated, common for BED, GTF attribute exports)
df = pd.read_csv('data.tsv', sep='\t')

# GTF / GFF (comment lines start with #)
df = pd.read_csv('genes.gtf', sep='\t', comment='#', header=None)

# Excel
df = pd.read_excel('samples.xlsx', sheet_name='Sheet1')

# Write back
df.to_csv('output.csv', index=False)
df.to_excel('output.xlsx', index=False)
```

We will practice file I/O by creating small in-memory datasets and round-tripping them.

In [ ]:
import io

# Simulate reading a CSV from a GEO-style expression file
csv_text = """gene_id,Sample_Control_1,Sample_Control_2,Sample_Treated_1,Sample_Treated_2
ENSG00000141510,120.5,135.2,340.1,355.8
ENSG00000146648,45.3,50.1,44.9,48.2
ENSG00000136997,300.0,280.4,310.2,295.5
ENSG00000141736,10.2,12.0,9.8,11.5
"""

geo_df = pd.read_csv(io.StringIO(csv_text), index_col='gene_id')
print("Loaded GEO-like dataset:")
print(geo_df)

In [ ]:
# Simulate a TSV sample metadata sheet
tsv_text = """sample_id\tcondition\tbatch\tage
Sample_Control_1\tcontrol\tA\t35
Sample_Control_2\tcontrol\tB\t42
Sample_Treated_1\ttreated\tA\t38
Sample_Treated_2\ttreated\tB\t40
"""

metadata = pd.read_csv(io.StringIO(tsv_text), sep='\t')
print("Sample metadata:")
print(metadata)

### 2.8 GroupBy

Split-apply-combine: group rows by a column, apply an aggregation, and combine results.

In [ ]:
# Create a larger gene table
np.random.seed(42)
gene_table = pd.DataFrame({
    'gene': [f'Gene_{i}' for i in range(20)],
    'chromosome': np.random.choice(['chr1', 'chr2', 'chr3', 'chrX'], 20),
    'biotype': np.random.choice(['protein_coding', 'lncRNA', 'pseudogene'], 20),
    'expression': np.random.lognormal(5, 1, 20).round(1),
})

print("Genes per chromosome:")
print(gene_table.groupby('chromosome')['gene'].count())

print("\nMean expression by biotype:")
print(gene_table.groupby('biotype')['expression'].mean().round(1))

In [ ]:
# Multiple aggregations at once
summary = gene_table.groupby('chromosome')['expression'].agg(
    ['count', 'mean', 'median', 'std']
).round(1)

print("Expression summary by chromosome:")
print(summary)

### 2.9 Merge and Join

Combining DataFrames is essential when your gene annotations, expression values, and sample metadata live in separate files.

In [ ]:
# Expression data
expr_data = pd.DataFrame({
    'gene_id': ['ENSG001', 'ENSG002', 'ENSG003', 'ENSG004', 'ENSG005'],
    'mean_expression': [150.5, 89.3, 245.8, 312.4, 10.2],
})

# Annotation (note: ENSG005 is missing, ENSG006 is extra)
annotation = pd.DataFrame({
    'gene_id': ['ENSG001', 'ENSG002', 'ENSG003', 'ENSG004', 'ENSG006'],
    'gene_name': ['BRCA1', 'TP53', 'EGFR', 'MYC', 'PTEN'],
    'chromosome': ['chr17', 'chr17', 'chr7', 'chr8', 'chr10'],
})

# Inner join -- only genes present in both tables
inner = pd.merge(expr_data, annotation, on='gene_id', how='inner')
print("Inner join (intersection):")
print(inner)

# Left join -- keep all expression rows, fill missing annotation with NaN
left = pd.merge(expr_data, annotation, on='gene_id', how='left')
print("\nLeft join (keep all expression rows):")
print(left)

In [ ]:
# Outer join -- keep everything
outer = pd.merge(expr_data, annotation, on='gene_id', how='outer')
print("Outer join (union):")
print(outer)

### 2.10 Biological Application: Gene Expression Analysis

Let us build a complete mini-analysis: generate an expression matrix, compute fold changes, and identify differentially expressed genes.

In [ ]:
# Generate expression data: 50 genes, 6 samples (3 control + 3 treatment)
np.random.seed(42)
n_genes = 50
genes = [f'Gene_{i:03d}' for i in range(n_genes)]
samples = ['Ctrl_1', 'Ctrl_2', 'Ctrl_3', 'Treat_1', 'Treat_2', 'Treat_3']

# Baseline expression
baseline = np.random.lognormal(mean=5, sigma=1, size=(n_genes, 6))

# Make 10 genes upregulated and 5 downregulated in treatment
up_idx = np.random.choice(n_genes, 10, replace=False)
down_idx = np.random.choice([i for i in range(n_genes) if i not in up_idx], 5, replace=False)

baseline[up_idx, 3:] *= 3.0    # 3-fold up
baseline[down_idx, 3:] *= 0.3  # ~3-fold down

expr_df = pd.DataFrame(baseline.round(1), columns=samples, index=genes)
print(f"Expression matrix: {expr_df.shape[0]} genes x {expr_df.shape[1]} samples")
print(expr_df.head())

In [ ]:
# Compute mean expression per condition
ctrl_cols = ['Ctrl_1', 'Ctrl_2', 'Ctrl_3']
treat_cols = ['Treat_1', 'Treat_2', 'Treat_3']

ctrl_mean = expr_df[ctrl_cols].mean(axis=1)
treat_mean = expr_df[treat_cols].mean(axis=1)

# Log2 fold change
log2fc = np.log2(treat_mean / ctrl_mean)

results = pd.DataFrame({
    'ctrl_mean': ctrl_mean.round(1),
    'treat_mean': treat_mean.round(1),
    'log2FC': log2fc.round(3),
})

print("Top 10 by absolute fold change:")
print(results.reindex(results['log2FC'].abs().sort_values(ascending=False).index).head(10))

In [ ]:
# Simple t-test to get p-values
from scipy import stats

pvalues = []
for gene in expr_df.index:
    _, pval = stats.ttest_ind(expr_df.loc[gene, ctrl_cols],
                              expr_df.loc[gene, treat_cols])
    pvalues.append(pval)

results['pvalue'] = pvalues
results['significant'] = (results['log2FC'].abs() > 1) & (results['pvalue'] < 0.05)

print(f"Significant genes (|log2FC| > 1, p < 0.05): {results['significant'].sum()}")
print("\nSignificant genes:")
print(results[results['significant']].sort_values('log2FC'))

### 2.11 Working with Sample Metadata and Count Matrices

In a real RNA-seq analysis you typically have two files: a count matrix (genes x samples) and a metadata table (samples x clinical variables). Here is how to work with them together.

In [ ]:
# Sample metadata
sample_meta = pd.DataFrame({
    'sample': samples,
    'condition': ['control']*3 + ['treatment']*3,
    'batch': ['A', 'B', 'A', 'A', 'B', 'A'],
    'rin_score': [8.5, 7.9, 8.2, 8.1, 7.5, 8.8],
})

print("Sample metadata:")
print(sample_meta)

In [ ]:
# Select only batch A samples from the expression matrix
batch_a = sample_meta[sample_meta['batch'] == 'A']['sample'].tolist()
print(f"Batch A samples: {batch_a}")
print(f"\nExpression for batch A (first 5 genes):")
print(expr_df[batch_a].head())

In [ ]:
# Mean expression per condition using metadata to select columns
for cond in ['control', 'treatment']:
    cols = sample_meta[sample_meta['condition'] == cond]['sample'].tolist()
    mean_expr = expr_df[cols].mean(axis=1)
    print(f"\nMean expression ({cond}), first 5 genes:")
    print(mean_expr.head().round(1))

---

## Summary

### NumPy

| Operation | Example |
|-----------|---------|
| Create array | `np.array([1,2,3])`, `np.zeros(10)`, `np.arange(100)` |
| Shape / reshape | `arr.shape`, `arr.reshape(3,4)`, `arr.flatten()` |
| Vectorized math | `a + b`, `np.log2(a)`, `a > 0` |
| Indexing | `arr[0]`, `arr[2:5]`, `arr[arr > 0]` |
| Statistics | `np.mean()`, `np.std()`, `np.percentile()` |
| Broadcasting | `matrix / row_vector` |
| Linear algebra | `np.dot()`, `np.corrcoef()`, `@` |

### Pandas

| Operation | Example |
|-----------|---------|
| Create DataFrame | `pd.DataFrame({'col': [1,2]})` |
| Read / write | `pd.read_csv()`, `df.to_csv()` |
| Select columns | `df['col']`, `df[['a','b']]` |
| Select rows | `df.loc[0]`, `df.iloc[:3]` |
| Filter | `df[df['col'] > 5]`, `df.query()` |
| Group | `df.groupby('col').mean()` |
| Merge | `pd.merge(df1, df2, on='key')` |
| Sort | `df.sort_values('col')` |

---

## Exercises

Work through the exercises below. Solutions are provided at the end of the notebook.

### Exercise 1: NumPy Array Operations

Given the following read counts and gene lengths, compute:
1. The TPM (Transcripts Per Million) for each gene. TPM formula:
   - Divide each gene's count by its length in kilobases to get RPK.
   - Divide each RPK by the sum of all RPKs, then multiply by 1,000,000.
2. The Pearson correlation between `counts_A` and `counts_B`.

In [ ]:
counts_A = np.array([500, 20, 1000, 300, 50])
counts_B = np.array([480, 25, 1050, 280, 55])
gene_lengths = np.array([2000, 500, 8000, 3000, 1000])  # in bp

# Your code here


### Exercise 2: Boolean Indexing

Given the array of GC content values, find:
1. How many genes have GC content above 50%.
2. The mean GC content of those high-GC genes.
3. The indices (positions) of genes with GC content below 40%.

In [ ]:
gc_values = np.array([0.42, 0.55, 0.38, 0.61, 0.47, 0.39, 0.52, 0.44, 0.58, 0.36])

# Your code here


### Exercise 3: Build a Gene DataFrame

Create a DataFrame with the following columns: `gene_name`, `chromosome`, `start`, `stop`, `strand`. Populate it with at least 6 genes of your choice (or invent them). Then:

1. Add a `length` column computed from start and stop.
2. Filter to genes on chromosome 17.
3. Sort by length descending.

In [ ]:
# Your code here


### Exercise 4: GroupBy on Expression Data

Using the `gene_table` DataFrame defined earlier in section 2.8:

1. Find the chromosome with the highest total expression.
2. For each biotype, compute the median and max expression.
3. Count the number of genes per chromosome-biotype combination (hint: group by two columns).

In [ ]:
# Your code here


### Exercise 5: Merge and Analyze

You have an expression table and a pathway annotation table. Merge them and answer:

1. Which pathway has the highest mean expression?
2. How many genes have no pathway annotation?
3. What is the mean expression of unannotated genes vs. annotated genes?

In [ ]:
expression = pd.DataFrame({
    'gene': ['BRCA1', 'TP53', 'EGFR', 'MYC', 'KRAS', 'PIK3CA', 'PTEN', 'RB1'],
    'expression': [150, 89, 245, 312, 178, 95, 63, 41],
})

pathways = pd.DataFrame({
    'gene': ['BRCA1', 'TP53', 'EGFR', 'MYC', 'KRAS', 'PIK3CA'],
    'pathway': ['DNA_Repair', 'Apoptosis', 'EGFR_Signaling', 'Cell_Cycle',
                'MAPK_Signaling', 'PI3K_Signaling'],
})

# Your code here


### Exercise 6: PWM Scoring

Using the `build_pwm` and `score_sequence` functions from section 1.8, create a PWM from these Sp1 binding site sequences and score the three candidate sequences below. Which one is most likely a real Sp1 site?

```
Sp1 sites: GGGCGG, GGGCGG, GGGCGA, GGGCGG, AGGCGG
Candidates: GGGCGG, AAATTT, GGGCGA
```

In [ ]:
# Your code here


---

## Solutions

### Solution 1

In [ ]:
counts_A = np.array([500, 20, 1000, 300, 50])
counts_B = np.array([480, 25, 1050, 280, 55])
gene_lengths = np.array([2000, 500, 8000, 3000, 1000])  # in bp

# TPM for sample A
rpk_A = counts_A / (gene_lengths / 1000)   # reads per kilobase
tpm_A = rpk_A / rpk_A.sum() * 1_000_000
print("TPM (sample A):", np.round(tpm_A, 1))

# TPM for sample B
rpk_B = counts_B / (gene_lengths / 1000)
tpm_B = rpk_B / rpk_B.sum() * 1_000_000
print("TPM (sample B):", np.round(tpm_B, 1))

# Pearson correlation
r = np.corrcoef(counts_A, counts_B)[0, 1]
print(f"Pearson r: {r:.4f}")

### Solution 2

In [ ]:
gc_values = np.array([0.42, 0.55, 0.38, 0.61, 0.47, 0.39, 0.52, 0.44, 0.58, 0.36])

# 1. Count genes with GC > 50%
high_gc = gc_values > 0.50
print(f"Genes with GC > 50%: {high_gc.sum()}")

# 2. Mean GC of high-GC genes
print(f"Mean GC of high-GC genes: {gc_values[high_gc].mean():.3f}")

# 3. Indices of low-GC genes (< 40%)
low_gc_idx = np.where(gc_values < 0.40)[0]
print(f"Indices of genes with GC < 40%: {low_gc_idx}")
print(f"Their GC values: {gc_values[low_gc_idx]}")

### Solution 3

In [ ]:
gene_df = pd.DataFrame({
    'gene_name':  ['BRCA1', 'TP53', 'EGFR', 'MYC', 'KRAS', 'HER2'],
    'chromosome':  [17, 17, 7, 8, 12, 17],
    'start':      [43044295, 7661779, 55019021, 127735434, 25204789, 37844393],
    'stop':       [43125483, 7687538, 55211628, 127742951, 25250936, 37884915],
    'strand':     ['-', '-', '+', '+', '-', '+'],
})

# 1. Add length
gene_df['length'] = gene_df['stop'] - gene_df['start'] + 1
print(gene_df)

# 2. Filter to chr 17
chr17 = gene_df[gene_df['chromosome'] == 17]
print("\nChromosome 17 genes:")
print(chr17[['gene_name', 'chromosome', 'length']])

# 3. Sort by length descending
print("\nSorted by length (descending):")
print(gene_df.sort_values('length', ascending=False)[['gene_name', 'length']])

### Solution 4

In [ ]:
# 1. Chromosome with highest total expression
total_by_chr = gene_table.groupby('chromosome')['expression'].sum()
print(f"Highest total expression: {total_by_chr.idxmax()} ({total_by_chr.max():.1f})")

# 2. Median and max expression by biotype
biotype_stats = gene_table.groupby('biotype')['expression'].agg(['median', 'max']).round(1)
print("\nExpression stats by biotype:")
print(biotype_stats)

# 3. Genes per chromosome-biotype combination
combo = gene_table.groupby(['chromosome', 'biotype']).size()
print("\nGenes per chromosome-biotype:")
print(combo)

### Solution 5

In [ ]:
expression = pd.DataFrame({
    'gene': ['BRCA1', 'TP53', 'EGFR', 'MYC', 'KRAS', 'PIK3CA', 'PTEN', 'RB1'],
    'expression': [150, 89, 245, 312, 178, 95, 63, 41],
})

pathways = pd.DataFrame({
    'gene': ['BRCA1', 'TP53', 'EGFR', 'MYC', 'KRAS', 'PIK3CA'],
    'pathway': ['DNA_Repair', 'Apoptosis', 'EGFR_Signaling', 'Cell_Cycle',
                'MAPK_Signaling', 'PI3K_Signaling'],
})

# Left merge to keep all expression rows
merged = pd.merge(expression, pathways, on='gene', how='left')

# 1. Pathway with highest mean expression
pathway_expr = merged.groupby('pathway')['expression'].mean()
print(f"Highest mean expression pathway: {pathway_expr.idxmax()} ({pathway_expr.max():.1f})")

# 2. Genes with no pathway annotation
no_pathway = merged[merged['pathway'].isna()]
print(f"\nGenes without pathway annotation: {len(no_pathway)}")
print(no_pathway[['gene', 'expression']])

# 3. Mean expression: annotated vs unannotated
annotated_mean = merged[merged['pathway'].notna()]['expression'].mean()
unannotated_mean = merged[merged['pathway'].isna()]['expression'].mean()
print(f"\nMean expression (annotated):   {annotated_mean:.1f}")
print(f"Mean expression (unannotated): {unannotated_mean:.1f}")

### Solution 6

In [ ]:
sp1_sites = ['GGGCGG', 'GGGCGG', 'GGGCGA', 'GGGCGG', 'AGGCGG']
sp1_pwm = build_pwm(sp1_sites)

print("Sp1 PWM:")
for i, nuc in enumerate('ATGC'):
    print(f"  {nuc}: {np.round(sp1_pwm[i], 2)}")

candidates = ['GGGCGG', 'AAATTT', 'GGGCGA']
print("\nCandidate scores:")
for seq in candidates:
    s = score_sequence(sp1_pwm, seq)
    print(f"  {seq}: {s:.2f}")

best = max(candidates, key=lambda s: score_sequence(sp1_pwm, s))
print(f"\nMost likely Sp1 site: {best}")

---

**Next module:** [Module 17 -- Data Wrangling](../17_Data_Wrangling/01_data_wrangling.ipynb)

---

[← Previous: Module 15: Error Handling and Debugging](../../Tier_1_Python_for_Bioinformatics/15_Error_Handling/01_error_handling.ipynb) | [Next: Module 17: Data Wrangling with Pandas →](../../Tier_1_Python_for_Bioinformatics/17_Data_Wrangling/01_data_wrangling.ipynb)